In [ ]:
import pandas as pd
import pyotp
from datetime import datetime
import time
import csv
import logging
import websocket as ws_client

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)


usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)


In [ ]:
### working tick data code

from datetime import datetime

feed_opened = False
feedJson = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']
            
            # Check if token exists in feedJson
            if token not in feedJson:
                feedJson[token] = []  # Create an empty list for new tokens

            # Append the new tick data to the list
            feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")  # Informative error message
    
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")
    

def open_callback():
    global feed_opened
    feed_opened = True


api.start_websocket( order_update_callback=event_handler_order_update,
                     subscribe_callback=event_handler_feed_update, 
                     socket_open_callback=open_callback)

while(feed_opened==False):
    pass

#subscribe to multiple tokens
api.subscribe(['MCX|428009','NSE|26009'])

In [ ]:
feedJson

In [ ]:
###### working resampling

import threading
import time
from datetime import datetime

import numpy as np

feed_opened = False
feedJson = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']
            
            # Check if token exists in feedJson
            if token not in feedJson:
                feedJson[token] = []  # Create an empty list for new tokens

            # Append the new tick data to the list
            feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")  # Informative error message
    
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")
    

def open_callback():
    global feed_opened
    feed_opened = True


# Assuming api.start_websocket(...) is correctly initialized elsewhere
api.start_websocket(order_update_callback=event_handler_order_update,
                    subscribe_callback=event_handler_feed_update, 
                    socket_open_callback=open_callback)

while not feed_opened:
    pass

# Subscribe to multiple tokens
api.subscribe(['MCX|428009', 'NSE|26009'])

# Event to signal thread to stop
stop_event = threading.Event()

last_resampled_data = {}

def resample_data(stop_event):
    while not stop_event.is_set():
        # Convert feedJson into a DataFrame
        df_list = []
        for token, data_list in feedJson.items():
            df_token = pd.DataFrame(data_list)
            df_token['tt'] = pd.to_datetime(df_token['tt'])  # Convert 'tt' column to datetime if needed
            df_list.append(df_token)
        
        if not df_list:
            continue
        
        df_combined = pd.concat(df_list, keys=feedJson.keys())  # Combine DataFrames with keys as tokens
        
        # Resample each token's data to 15-second intervals
        resampled_data = df_combined.groupby(level=0).apply(lambda x: x.set_index('tt').resample('15s').agg({'ltp': 'ohlc'}))
        
        # Process resampled_data as needed (e.g., store or print)
        print(resampled_data)
        
        time.sleep(5)  # Adjust interval as needed for checking new data


# Start the thread to print feedJson
thread_resample = threading.Thread(target=resample_data, args=(stop_event,), daemon=True)
thread_resample.start()


In [ ]:
###### working resampling and ema

import threading
import time
from datetime import datetime
import pandas as pd
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)


feed_opened = False
feedJson = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']
            
            # Check if token exists in feedJson
            if token not in feedJson:
                feedJson[token] = []  # Create an empty list for new tokens

            # Append the new tick data to the list
            feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")  # Informative error message
    
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")
    

def open_callback():
    global feed_opened
    feed_opened = True


# Assuming api.start_websocket(...) is correctly initialized elsewhere
api.start_websocket(order_update_callback=event_handler_order_update,
                    subscribe_callback=event_handler_feed_update, 
                    socket_open_callback=open_callback)

while not feed_opened:
    pass

# Subscribe to multiple tokens
api.subscribe(['MCX|428009', 'NSE|26009'])

# Event to signal thread to stop
stop_event = threading.Event()

last_resampled_data = {}

def resample_data(stop_event):
    while not stop_event.is_set():
        try:
            # Convert feedJson into a DataFrame
            df_list = []
            for token, data_list in feedJson.items():
                try:
                    df_token = pd.DataFrame(data_list)
                    df_token['tt'] = pd.to_datetime(df_token['tt'])  # Convert 'tt' column to datetime if needed
                    df_list.append((token, df_token))
                except Exception as e:
                    logging.error(f"Error processing token {token} data: {e}")
                    continue
            
            if not df_list:
                continue
            
            # Combine DataFrames with keys as tokens
            resampled_data = {}
            for token, df_token in df_list:
                try:
                    df_token = df_token.set_index('tt').resample('15s').agg({'ltp': 'ohlc'})
                    resampled_data[token] = df_token
                except Exception as e:
                    logging.error(f"Error resampling token {token} data: {e}")
                    continue
            
            # Compute EMAs for each token's resampled data
            for token, df in resampled_data.items():
                try:
                    df['EMA1'] = df['ltp']['close'].ewm(span=3, adjust=False).mean()
                    df['EMA2'] = df['ltp']['close'].ewm(span=10, adjust=False).mean()


                    last_resampled_data[token] = df
                    
                    # Store or print the resampled data with EMAs
                    logging.info(f"Token {token}: Resampled Data with EMAs\n{df}")
                except Exception as e:
                    logging.error(f"Error computing EMA for token {token}: {e}")
                    continue
        
        except Exception as e:
            logging.error(f"Error in resample_data: {e}")
        
        time.sleep(5)  # Adjust interval as needed for checking new data


# Start the thread to print feedJson
thread_resample = threading.Thread(target=resample_data, args=(stop_event,), daemon=True)
thread_resample.start()


In [ ]:
import csv

In [ ]:
############## TRADE LOGIC

import threading
import time
from datetime import datetime
import pandas as pd
import numpy as np
import csv

feed_opened = False
feedJson = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']
            
            # Check if token exists in feedJson
            if token not in feedJson:
                feedJson[token] = []  # Create an empty list for new tokens

            # Append the new tick data to the list
            feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")  # Informative error message
    
def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")
    

def open_callback():
    global feed_opened
    feed_opened = True


# Assuming api.start_websocket(...) is correctly initialized elsewhere
api.start_websocket(order_update_callback=event_handler_order_update,
                    subscribe_callback=event_handler_feed_update, 
                    socket_open_callback=open_callback)

while not feed_opened:
    pass

# Subscribe to multiple tokens
api.subscribe(['MCX|428009', 'NSE|26009'])

# Event to signal thread to stop
stop_event = threading.Event()

last_resampled_data = {}

def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(trade_log)

def resample_data(stop_event):
    while not stop_event.is_set():
        try:
            df_list = []
            for token, data_list in feedJson.items():
                df_token = pd.DataFrame(data_list)
                df_token['tt'] = pd.to_datetime(df_token['tt'])  # Convert 'tt' column to datetime if needed
                df_token.set_index('tt', inplace=True)
                df_token = df_token.resample('15s').agg({'ltp': 'ohlc'}).dropna()  # Resample and drop NaN values
                df_list.append((token, df_token))
            
            if not df_list:
                continue
            
            # Calculate EMA1 and EMA2
            for token, df in df_list:
                df['ema1'] = df['ltp']['close'].ewm(span=3, adjust=False).mean()
                df['ema2'] = df['ltp']['close'].ewm(span=10, adjust=False).mean()
                
                # Update last_resampled_data for the token
                last_resampled_data[token] = df
            
            # Process resampled_data as needed (e.g., store or print)
            #print(last_resampled_data)
            
            time.sleep(5)  # Adjust interval as needed for checking new data
        
        except Exception as e:
            print(f"Error in resample_data: {e}")  # Error handling

def trading_logic(stop_event):
    current_positions = {}

    while not stop_event.is_set():
        try:
            for token, df in last_resampled_data.items():
                if 'ema1' in df.columns and 'ema2' in df.columns:
                    # Check for crossover conditions
                    if df['ema1'].iloc[-1] > df['ema2'].iloc[-1] and df['ema1'].iloc[-2] <= df['ema2'].iloc[-2]:
                        if token not in current_positions or current_positions[token] == 'short':
                            # Exit short position (if exists)
                            if token in current_positions and current_positions[token] == 'short':
                                exit_timestamp = df.index[-1]
                                exit_price = df['ltp']['close'].iloc[-1]  # Corrected to access 'close' price under 'ltp'
                                print(f"Exit Short: {token} at {exit_timestamp}, Price: {exit_price}")
                                exit_log = ["Exit Short", token, exit_timestamp, exit_price]
                                append_to_csv(exit_log)


                            # Enter long position
                            entry_timestamp = df.index[-1]
                            entry_price = df['ltp']['close'].iloc[-1]  # Corrected to access 'close' price under 'ltp'
                            print(f"Enter Long: {token} at {entry_timestamp}, Price: {entry_price}")
                            entry_log = ["Enter Long", token, entry_timestamp, entry_price]
                            append_to_csv(entry_log)


                            # Update current position
                            current_positions[token] = 'long'

                    elif df['ema1'].iloc[-1] < df['ema2'].iloc[-1] and df['ema1'].iloc[-2] >= df['ema2'].iloc[-2]:
                        if token not in current_positions or current_positions[token] == 'long':
                            # Exit long position (if exists)
                            if token in current_positions and current_positions[token] == 'long':
                                exit_timestamp = df.index[-1]
                                exit_price = df['ltp']['close'].iloc[-1]  # Corrected to access 'close' price under 'ltp'
                                print(f"Exit Long: {token} at {exit_timestamp}, Price: {exit_price}")
                                exit_log = ["Exit Long", token, exit_timestamp, exit_price]
                                append_to_csv(exit_log)


                            # Enter short position
                            entry_timestamp = df.index[-1]
                            entry_price = df['ltp']['close'].iloc[-1]  # Corrected to access 'close' price under 'ltp'
                            print(f"Enter Short: {token} at {entry_timestamp}, Price: {entry_price}")
                            entry_log = ["Enter Short", token, entry_timestamp, entry_price]
                            append_to_csv(entry_log)


                            # Update current position
                            current_positions[token] = 'short'

            time.sleep(1)  # Adjust interval as needed for checking trading signals

        except Exception as e:
            print(f"Error in trading_logic: {e}")  # Error handling



# Start the resampling thread
thread_resample = threading.Thread(target=resample_data, args=(stop_event,), daemon=True)
thread_resample.start()

# Start the trading logic thread
thread_trade = threading.Thread(target=trading_logic, args=(stop_event,), daemon=True)
thread_trade.start()


In [ ]:

api.close_websocket()
# Stop the thread gracefully by setting the event
stop_event.set()
print("Thread stopped gracefully")

In [ ]:
feedJson

In [3]:
resample_data

NameError: name 'resample_data' is not defined

In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import xarray as xr

feed_opened = False
feedJson = {}
resample_frequency = "1min"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token not in feedJson:
                    feedJson[token] = []
                    last_resample_time[token] = timest

                feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    print(f"Initial tick data for token {token}: {ticks}")

                    new_da = xr.DataArray(
                        [tick["ltp"] for tick in ticks],
                        dims="time",
                        coords={"time": pd.to_datetime([tick["tt"] for tick in ticks])},
                    )

                    print(f"New DataArray for token {token}:\n{new_da}")

                    if token in resampled_data:
                        da = resampled_data[token]
                        last_resampled_time = da.time.max().values
                        new_da = new_da.sel(time=slice(last_resampled_time, None))

                        print(f"Filtered new DataArray for token {token}:\n{new_da}")
                    else:
                        da = new_da

                    if len(new_da['time']) > 0:
                        da = xr.concat([da, new_da], dim="time")
                        da = da.groupby("time").mean()

                        print(f"Concatenated DataArray for token {token}:\n{da}")

                    resampled_da = da.resample(time=resample_frequency).ohlc()

                    print(f"Resampled data for token {token}:\n{resampled_da}")

                    resampled_data[token] = resampled_da
                    last_resample_time[token] = resampled_da.time.max().values

                    feedJson[token] = []
                except Exception as e:
                    print(f"Error resampling data for token {token}: {e}")
        await asyncio.sleep(1)

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

api.start_websocket(
    order_update_callback=event_handler_order_update,
    subscribe_callback=event_handler_feed_update,
    socket_open_callback=open_callback
)

while not feed_opened:
    pass

api.subscribe(['MCX|428009', 'NSE|26009'])

async def main():
    await resample_ticks()

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    loop.create_task(main())
else:
    loop.run_until_complete(main())
